In [1]:
!nvidia-smi

Fri Jan 31 01:27:47 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


## Add Matrix

In [21]:
%%writefile matrixAdd.cu
#include <stdio.h>

// Kernel function to add two matrices
__global__ void matrixAdd(float *A, float *B, float *C, int numRows, int numCols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < numRows && col < numCols) {
        int index = row * numCols + col;
        C[index] = A[index] + B[index];
    }
}

void printMatrix(float *matrix, int numRows, int numCols) {
    for (int i = 0; i < numRows; ++i) {
        for (int j = 0; j < numCols; ++j) {
            printf("%.2f ", matrix[i * numCols + j]);
        }
        printf("\n");
    }
}

int main() {
    // Matrix dimensions
    int numRows = 3;  // Reduced for easier printing
    int numCols = 3;  // Reduced for easier printing
    int size = numRows * numCols * sizeof(float);

    // Host matrices
    float *h_A, *h_B, *h_C;

    // Allocate host memory
    h_A = (float*)malloc(size);
    h_B = (float*)malloc(size);
    h_C = (float*)malloc(size);

    // Initialize host matrices
    for (int i = 0; i < numRows * numCols; ++i) {
        h_A[i] = 1.0f;
        h_B[i] = 2.0f;
    }

    // Device matrices
    float *d_A, *d_B, *d_C;

    // Allocate device memory
    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    // Copy host matrices to device
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Define grid and block dimensions
    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks((numCols + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (numRows + threadsPerBlock.y - 1) / threadsPerBlock.y);

    // Launch the kernel
    matrixAdd<<<numBlocks, threadsPerBlock>>>(d_A, d_B, d_C, numRows, numCols);

    // Copy the result back to the host
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    // Print the matrices
    printf("Matrix A:\n");
    printMatrix(h_A, numRows, numCols);

    printf("\nMatrix B:\n");
    printMatrix(h_B, numRows, numCols);

    printf("\n C  = (A + B):\n");
    printMatrix(h_C, numRows, numCols);

    // Free device memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    // Free host memory
    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Overwriting matrixAdd.cu


In [19]:
!nvcc -arch=compute_70 -code=sm_70 matrixAdd.cu -o matAdd

In [20]:
! ./matAdd

Matrix A:
1.00 1.00 1.00 
1.00 1.00 1.00 
1.00 1.00 1.00 

Matrix B:
2.00 2.00 2.00 
2.00 2.00 2.00 
2.00 2.00 2.00 

Resultant Matrix C (A + B):
3.00 3.00 3.00 
3.00 3.00 3.00 
3.00 3.00 3.00 


## ReLU and GeLU

GELU(x) = 0.5 * x * (1 + erf(x / sqrt(2)))

GELU(x) ≈ 0.5 * x * (1 + tanh(sqrt(2/π) * (x + 0.044715 * x^3)))



Total number of blocks needed:


(ARRAY_SIZE + blockSize - 1) / blockSize


 ensures that we have enough blocks to cover all elements, even if ARRAY_SIZE is not perfectly divisible by blockSize.


 256 threads per block often allows for good occupancy (utilization of the GPU's compute resources) without exceeding the maximum number of threads per block for most GPUs.

In [13]:
%%writefile activation_functions.cu
#include <stdio.h>
#include <math.h>

#define ARRAY_SIZE 5

// ReLU activation function kernel
__global__ void relu(float *input, float *output, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        output[idx] = fmaxf(0.0f, input[idx]);
    }
}

// GELU activation function kernel
__global__ void gelu(float *input, float *output, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        float x = input[idx];
        output[idx] = 0.5f * x * (1.0f + tanhf(sqrtf(2.0f / M_PI) * (x + 0.044715f * powf(x, 3))));
    }
}

int main() {
    float h_input[ARRAY_SIZE] = {-2.0f, -0.5f, 0.0f, 0.5f, 2.5f};
    float h_output_relu[ARRAY_SIZE];
    float h_output_gelu[ARRAY_SIZE];

    float *d_input, *d_output;
    int size = ARRAY_SIZE * sizeof(float);

    // Allocate device memory
    cudaMalloc((void**)&d_input, size);
    cudaMalloc((void**)&d_output, size);

    // Copy input data to device
    cudaMemcpy(d_input, h_input, size, cudaMemcpyHostToDevice);

    // Define grid and block dimensions
    int blockSize = 256;
    int gridSize = (ARRAY_SIZE + blockSize - 1) / blockSize;

    // Apply ReLU
    relu<<<gridSize, blockSize>>>(d_input, d_output, ARRAY_SIZE);
    cudaMemcpy(h_output_relu, d_output, size, cudaMemcpyDeviceToHost);

    // Apply GELU
    gelu<<<gridSize, blockSize>>>(d_input, d_output, ARRAY_SIZE);
    cudaMemcpy(h_output_gelu, d_output, size, cudaMemcpyDeviceToHost);

    // Print results
    printf("Input\t\tReLU\t\tGELU\n");
    for (int i = 0; i < ARRAY_SIZE; i++) {
        printf("%f\t%f\t%f\n", h_input[i], h_output_relu[i], h_output_gelu[i]);
    }

    // Free device memory
    cudaFree(d_input);
    cudaFree(d_output);

    return 0;
}

Overwriting activation_functions.cu


In [14]:
!nvcc -arch=compute_70 -code=sm_70 activation_functions.cu -o actFN

In [15]:
! ./actFN

Input		ReLU		GELU
-2.000000	0.000000	-0.045402
-0.500000	0.000000	-0.154286
0.000000	0.000000	0.000000
0.500000	0.500000	0.345714
2.500000	2.500000	2.484916
